# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os
import shutil
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('data'):
    shutil.rmtree('data')
    os.makedirs('data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    CatsUserID = 2496, 
    days_back = 365 * 2,
    out_of_scope = ['-80', 'Cryo tank', 'Water'],
    coris_enabled = False,
    hobolink_enabled = True,
    conserv_enabled = False, # out of scope for this project (adding Hobolink). 
    testing = True
)

DEBUG: Enabled data sources: ['Hobolink']


Gathering Hobolink readings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:06<00:00,  1.64s/it]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-11-04 16:25:46,586 - EnvironmentData - INFO - Created Hobolink client
2025-11-04 16:25:46,587 - EnvironmentData - INFO - Enabled data sources: ['Hobolink']
2025-11-04 16:25:46,587 - EnvironmentData - WARNING - Date range (730 days) exceeds Hobolink API limit of 364 days. Capping to most recent 364 days.
2025-11-04 16:25:46,587 - EnvironmentData - INFO - Adjusted start_utc from 1699226746 to 1730849146
2025-11-04 16:25:46,587 - EnvironmentData - INFO - Using date range: 364 days of historical data
2025-11-04 16:25:46,587 - EnvironmentData - INFO - get_devices_as_dataframe
2025-11-04 16:25:46,587 - EnvironmentData - INFO - get_devices
2025-11-04 16:25:46,587 - EnvironmentData - INFO - Making API request to: devices
2025-11-04 16:25:46,587 - EnvironmentData - DEBUG - Parameters: {'includeSensors': 'true'}
2025-11-04 16:25:47,506 - EnvironmentData - INFO - Found 16 devices


This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
#pl.read_parquet('data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()
polars.read_parquet('data/sensor_readings.parquet').head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,str,str,str,str,str,str,f32,f32
1736967600,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-1""",null,"""Temperature""",72.458656,null
1736969400,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-1""",null,"""Temperature""",70.103424,null
1736345700,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179174-1""",null,"""Temperature""",68.674843,null
1736374500,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179174-1""",null,"""Temperature""",66.975983,null
1736377200,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179174-1""",null,"""Temperature""",70.528137,null


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [5]:
envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('data/new-readings')[0]
print(filename)
polars.read_parquet('data/new-readings/' + filename).sample(5)

1762298754.parquet


SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,str,str,str,str,str,str,f32,f32
1762298754,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.914246,null
1762298754,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-1""","""RX Station 2_Temperature""","""Temperature""",70.103424,null
1762298754,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-2""","""RX Station 2_RH""","""RH""",null,10.269321
1762298754,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.493401,null
1762298754,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,43.894104


# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [6]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
print(os.listdir('data/new-readings'))

C:\Users\Bryce\Documents\arbaiza\environmental-sensor-poc\EnvironmentData.py:220: UserWarning: consolidate_readings validation errors : Count of sensors missing from historical data: 38.

  warnings.warn(msg)


ColumnNotFoundError: unable to find column "QueryUTC"; valid columns: ["SensorReadingUTC", "Source", "DeviceID", "DeviceName", "SensorID", "SensorName", "SensorType", "SensorReadingF", "SensorReadingRh", "SensorReadingUTC_SecondsFromPrior"]

Let's look at the data we have now:

In [ ]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('data/sensor_readings.parquet')
sensor_readings.head()

In [ ]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type\.
sensor_readings.tail()

In [ ]:
# Device Readings.
device_readings = polars.read_parquet('data/device_readings.parquet')
device_readings.head()

In [ ]:
# Sensors
sensors = polars.read_parquet('data/sensors.parquet')
sensors.head()

In [ ]:
# Devices. 
devices = polars.read_parquet('data/sensors.parquet')
devices.head()

In [ ]:
# UTC Date/Time Info
utcs = polars.read_parquet('data/utcs.parquet').head()
utcs.head()

In [ ]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

In [ ]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('data/device_readings_daily.parquet')
device_readings_daily.head()

Once we are done working with data intake/processing, we close the class to release lgo file connections:

In [ ]:
# When done, close the connection to the logs. 
envdt.close()

# 4 Analytics

Now we are ready to pull data and run analytics. Here are a few examples: 

In [ ]:
# Histogram of temperature. 
import seaborn
import duckdb
dt = duckdb.sql("""
    SELECT SensorReadingF 
    FROM read_parquet('data/sensor_readings.parquet') 
    WHERE SensorReadingF is not null and SensorID_Coris = 21373;
""")
seaborn.histplot(dt.fetchnumpy())

In [ ]:
# Time series plot. 
dt = duckdb.sql("""
    SELECT SensorReadingUTC, SensorReadingF 
    FROM read_parquet('data/sensor_readings.parquet') 
    WHERE SensorReadingF is not null and SensorID_Coris = 21373;
""")
seaborn.lineplot(x = 'SensorReadingUTC', y= 'SensorReadingF', data=dt.fetchnumpy())

In [ ]:
# Time series plot, queried from Devices instead of Sensors. 
# This is the same data, just organized in a different way. 
dt = duckdb.sql("""
    SELECT SensorReadingUTC, SensorReadingF 
    FROM read_parquet('data/device_readings.parquet') 
    WHERE SensorReadingF is not null and DeviceID_Coris = 12162;
""")
seaborn.lineplot(x = 'SensorReadingUTC', y= 'SensorReadingF', data=dt.fetchnumpy())

In [ ]:
# Differentiate historical vs. cron readings by filtering on QueryUTC = NULL.
duckdb.sql("""
    SELECT *
    FROM read_parquet('data/device_readings.parquet') 
    WHERE QueryUTC is null
""")